# Sampling with Hyformer

This notebook is a walk-through of sampling with Hyformer.

You will implement a simple [rejection sampling](https://en.wikipedia.org/wiki/Rejection_sampling) based procedure and explore its limitations.

## Setup

Run this cell first, then run the rest of the notebook top to bottom. It clones the repository, installs the
package and downloads the model checkpoints. It takes about a minute and only needs to run once per session.

In [ ]:
# Setup: clone the repository, install the package, download the checkpoints
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/szczurek-lab/mlinpl-jsi.git"
CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/1yG0Qp3sVtrY_chM5L78V6QyfYjgX9CLl"


def pip_run(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", *args])


if not Path("../configs").is_dir():  # not inside the repository yet
    if not Path("mlinpl-jsi").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    # Colab already ships torch, numpy, pandas and scikit-learn.
    # --no-deps keeps those versions instead of reinstalling a multi-gigabyte torch.
    pip_run("pip", "install", "-q", "rdkit", "loguru", "gdown")
    pip_run("pip", "install", "-q", "--no-deps", "-e", "mlinpl-jsi")
    os.chdir("mlinpl-jsi/notebooks")  # every path below is relative to REPO_ROOT = cwd.parent

# An editable install registers itself through a .pth file, which Python only reads when it starts up.
# This kernel is already running, so point it at the sources directly instead of restarting the runtime.
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

# The checkpoints are distributed over Google Drive, not in the repository
if not Path("../model_ckpt/model_finetuned.pt").exists():
    pip_run("gdown", "--folder", "--quiet", CKPT_FOLDER_URL, "-O", "../model_ckpt")

import joint_improvement  # noqa: E402
import torch  # noqa: E402

print(f"working directory: {Path.cwd()}")
print(f"joint_improvement: {joint_improvement.__file__}")
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")
print("Setup finished. Continue to the next cell.")

In [ ]:
# Imports and repo root

from dataclasses import replace
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

from joint_improvement.hyformer import Hyformer, HyformerConfig
from joint_improvement.tokenizers import SMILESTokenizer
from joint_improvement.trainers import MultiTaskTrainer
from joint_improvement.trainers.multitask import TrainerConfig
from joint_improvement.utils.chemistry import calculate_logp, calculate_validity
from joint_improvement.utils.data.collators import LMCollator, PredictionCollator
from joint_improvement.utils.data.data_io import load_npz_data, save_npz_data
from joint_improvement.utils.data.dataloader import SequenceDataLoader
from joint_improvement.utils.data.dataset import SequenceDataset

REPO_ROOT = Path.cwd().parent  # notebook is run from notebooks/

In [ ]:
# Set the device and run a quick installation check: repo root, torch/device, and a logP sanity value for ethanol
device = "cuda" if torch.cuda.is_available() else "cpu"

assert (REPO_ROOT / "configs").is_dir(), f"REPO_ROOT looks wrong: {REPO_ROOT}"
print(f"torch {torch.__version__}, device: {device}")
print(f"logP(ethanol) = {calculate_logp('CCO'):.4f}")

## ZINC250k dataset

You will first assemble datasets of molecular compounds together with their [logP](https://en.wikipedia.org/wiki/Partition_coefficient) values (the octanol–water partition coefficient, a measure of lipophilicity).

ZINC250k is a random subset of ~250,000 drug-like molecules from the ZINC database, distributed with precomputed logP, QED and SAS values. It was popularized by the Chemical VAE (Gómez-Bombarelli et al., 2018) and has since become a standard benchmark in molecular machine learning, especially for generative models and property optimization (e.g. logP, QED). Its moderate size and clean, easy-to-parse SMILES make it a convenient testbed for comparing methods and for fast prototyping.

First, download ZINC250k (SMILES with precomputed properties) to `data/zinc250k/raw/`.

Then sample three logP datasets from it and save them to `data/zinc250k/logp/`:

- `random_1k`: 1,000 random molecules.
- `low_logp_1k`: 1,000 random molecules with logP below the dataset 25th percentile.
- `eval_100`: 100 held-out molecules for evaluation, disjoint from the two sets above.

In [ ]:
# Download the ZINC250k CSV (skipped if already present)
ZINC_URL = (
    "https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/"
    "models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv"
)
raw_path = REPO_ROOT / "data/zinc250k/raw/250k_rndm_zinc_drugs_clean_3.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_path.exists():
    urlretrieve(ZINC_URL, raw_path)
raw_path

In [ ]:
# Plot the logP distribution of ZINC250k with its 25th percentile
logp_all = pd.read_csv(raw_path)["logP"]
q25_logp = logp_all.quantile(0.25)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(logp_all, bins=100, color="steelblue", alpha=0.8)
ax.axvline(q25_logp, color="crimson", linestyle="--", label=f"25th percentile = {q25_logp:.2f}")
ax.set_xlabel("logP")
ax.set_ylabel("Count")
ax.set_title("ZINC250k logP distribution")
ax.legend()
plt.show()

In [ ]:
# Create 3 datasets: 1k random, 1k below-25th-percentile logP, 100 held-out for evaluation (disjoint from the first two)
df = pd.read_csv(raw_path)
smiles_all = df["smiles"].str.strip().to_numpy()  # SMILES in this CSV carry a trailing newline
logp_all = df["logP"].to_numpy()
q25_logp = np.percentile(logp_all, 25)  # our threshold for low logP molecules

rng = np.random.default_rng(0)
out_dir = REPO_ROOT / "data/zinc250k/logp"
out_dir.mkdir(parents=True, exist_ok=True)  # save_npz_data does not create the directory

random_idx = rng.choice(len(df), size=1000, replace=False)
low_idx = rng.choice(np.flatnonzero(logp_all < q25_logp), size=1000, replace=False)
eval_pool = np.setdiff1d(np.arange(len(df)), np.union1d(random_idx, low_idx))
eval_idx = rng.choice(eval_pool, size=100, replace=False)

for name, idx in [("random_1k", random_idx), ("low_logp_1k", low_idx), ("eval_100", eval_idx)]:
    # explicit str dtype: an object array would need pickle to load
    sequences = np.asarray(smiles_all[idx], dtype=str)
    save_npz_data(out_dir / f"{name}.npz", {"sequence": sequences, "labels": logp_all[idx]})
    print(f"{name}: n={len(idx)}, mean logP={logp_all[idx].mean():.3f}")

In [ ]:
# Plot the logP distributions of the three datasets on a single axis
bins = np.linspace(logp_all.min(), logp_all.max(), 50)

fig, ax = plt.subplots(figsize=(6, 4))
for name, idx in [("random_1k", random_idx), ("low_logp_1k", low_idx), ("eval_100", eval_idx)]:
    # density=True makes the 1k and 100-sample sets comparable
    ax.hist(logp_all[idx], bins=bins, density=True, alpha=0.5, label=f"{name} (n={len(idx)})")
ax.axvline(q25_logp, color="black", linestyle="--", label=f"25th percentile = {q25_logp:.2f}")
ax.set_xlabel("logP")
ax.set_ylabel("Density")
ax.set_title("logP distribution of the sampled datasets")
ax.legend()
plt.show()

## Hyformer: generate and predict with a single model 

Initialized with the weights pretrained in `00-pretrain.ipynb` (`results/zinc250k/pretrain/model.pt`).

In [ ]:
# Define the constants of this section
TOKENIZER_CONFIG_PATH = REPO_ROOT / "configs/tokenizers/smiles/zinc250k/tokenizer_config.json"
VOCAB_PATH = REPO_ROOT / "data/vocabularies/zinc250k.txt"
MODEL_CONFIG_PATH = REPO_ROOT / "configs/hyformer/base_8m.json"

In [ ]:
# Load the tokenizer, initialize the base model (with a logP regression head), load the pretrained weights and print its size
PRETRAINED_CKPT_PATH = REPO_ROOT / "model_ckpt/model.pt"

tokenizer = SMILESTokenizer.from_pretrained(TOKENIZER_CONFIG_PATH, vocab_filepath=VOCAB_PATH)

config = HyformerConfig.from_pretrained(MODEL_CONFIG_PATH)
model_config = replace(config, num_prediction_tasks=1, prediction_task_type="regression")
model = Hyformer.from_config(model_config).to(device).eval()  # from_config adds generate() (see generator_type)
model.load_pretrained(PRETRAINED_CKPT_PATH, device=device)  # non-strict: the regression head is new and stays randomly initialized

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params:,} ({n_params / 1e6:.2f}M)")

In [ ]:
# Jointly finetune Hyformer (causal LM + logP regression) on one dataset; the eval set drives early stopping
TRAIN_DATASET = "random_1k"
# TRAIN_DATASET = "low_logp_1k"
EVAL_DATASET = "eval_100"
TRAINER_CONFIG_PATH = REPO_ROOT / "configs/trainers/finetune.json"  # defines the task weights (lm / prediction)
FINETUNE_DIR = REPO_ROOT / "results/zinc250k/finetune"  # the best checkpoint (lowest eval loss) is saved here

trainer_config = TrainerConfig.from_pretrained(TRAINER_CONFIG_PATH)
COLLATORS = {"lm": LMCollator, "prediction": PredictionCollator}


def make_loader(name: str, task: str, shuffle: bool) -> SequenceDataLoader:
    data = load_npz_data(out_dir / f"{name}.npz", sequence_key="sequence", target_key="labels")
    return SequenceDataLoader(
        SequenceDataset(data["sequence"], targets=data["labels"]),
        COLLATORS[task](tokenizer, max_length=config.to_dict()["max_seq_len"]),
        batch_size=trainer_config.batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
    )


trainer = MultiTaskTrainer(trainer_config, model, device=device, out_dir=FINETUNE_DIR)
trainer.train(
    train_loaders={task: make_loader(TRAIN_DATASET, task, shuffle=True) for task in trainer_config.tasks},
    val_loaders={task: make_loader(EVAL_DATASET, task, shuffle=False) for task in trainer_config.tasks},
)  # the best weights are reloaded into `model` at the end

print(f"Best weighted eval loss: {trainer.state.best_val_loss:.4f} (epoch {trainer.state.best_epoch})")

In [ ]:
# Generate SMILES: start every sequence from [<lm> task token, <bos>] and sample token by token
# Sampling hyperparameters
N_GENERATED = 32  # number of sequences sampled in parallel
MAX_SEQUENCE_LENGTH = config.max_seq_len  # prefix + generated tokens
TEMPERATURE = 0.9  # <1 = more conservative samples, >1 = more diverse
TOP_K = 25  # sample only from the k most likely tokens (None = off)
TOP_P = None  # nucleus sampling: smallest token set with cumulative probability >= TOP_P (None = off)
SEED = 0  # seed of the sampling random number generator
N_TO_PLOT = 4  # stop printing/drawing after this many valid molecules

model.eval()
generator = torch.Generator(device=device).manual_seed(SEED)
prefix_ids = torch.tensor([[tokenizer.task_token_ids["lm"], tokenizer.bos_token_id]] * N_GENERATED, device=device)
generated_ids = model.generate(
    prefix_input_ids=prefix_ids,
    max_sequence_length=MAX_SEQUENCE_LENGTH - prefix_ids.shape[1],
    eos_token_id=tokenizer.eos_token_id,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    generator=generator,
)

generated_smiles = []
for ids in generated_ids[:, prefix_ids.shape[1] :].tolist():
    if tokenizer.eos_token_id in ids:  # sequences that finished early keep sampling, so cut at the first <eos>
        ids = ids[: ids.index(tokenizer.eos_token_id)]
    generated_smiles.append(tokenizer.decode(ids))

n_plotted = 0
for smiles in generated_smiles:
    is_valid = calculate_validity(smiles)
    print(f"valid={is_valid}  {smiles}")
    if is_valid:  # invalid SMILES cannot be drawn
        display(Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(250, 200)))
        n_plotted += 1
        if n_plotted == N_TO_PLOT:
            break


In [ ]:
# Predict logP of the first N_TO_PLOT valid generated molecules: tokenize with the <prediction> task token and run the regression head
valid_generated = [s for s in generated_smiles if calculate_validity(s)][:N_TO_PLOT]  # RDKit cannot score invalid SMILES

if not valid_generated:
    print("No valid generated molecules to predict")
else:
    model.eval()
    batch = PredictionCollator(tokenizer, max_length=config.max_seq_len)([{"sequence": s} for s in valid_generated])
    predicted_logp = model.predict(batch.input_ids.to(device), batch.attention_mask.to(device)).squeeze(-1).cpu().numpy()

    for smiles, pred in zip(valid_generated, predicted_logp):
        print(f"predicted={pred:6.2f}  true (RDKit)={calculate_logp(smiles):6.2f}  {smiles}")
        display(Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(250, 200)))


## Sample molecules

In [ ]:
# Sample molecules with a target logP: generate candidates once, then select with best-of-n and with rejection sampling
# TASK: generate molecules whose logP is close to TARGET_LOGP. Hyformer proposes candidates (generation) and scores
# them (logP prediction); an `accept_fn` then decides which candidates to keep:
#   - best-of-n: keep the single candidate whose predicted logP is closest to the target
#   - rejection sampling: keep every candidate whose predicted logP is within EPSILON of the target
# Evaluation of both methods is in the next cell.

# Task
TARGET_LOGP = float(np.percentile(logp_all, 95))  # target: the 95th percentile of the ZINC250k logP distribution
EPSILON = 0.5  # rejection sampling accepts a molecule if |predicted logP - target| <= EPSILON
N_CANDIDATES = 100  # number of molecules generated before selection

# Sampling hyperparameters
MAX_SEQUENCE_LENGTH = config.max_seq_len  # prefix + generated tokens
TEMPERATURE = 0.9  # <1 = more conservative samples, >1 = more diverse
TOP_K = 25  # sample only from the k most likely tokens (None = off)
TOP_P = None  # nucleus sampling: smallest token set with cumulative probability >= TOP_P (None = off)
SEED = 0  # seed of the sampling random number generator


def generate_candidates(n: int) -> list[str]:
    """Sample `n` SMILES from the Hyformer language model."""
    generator = torch.Generator(device=device).manual_seed(SEED)
    model.eval()
    prefix_ids = torch.tensor([[tokenizer.task_token_ids["lm"], tokenizer.bos_token_id]] * n, device=device)
    generated_ids = model.generate(
        prefix_input_ids=prefix_ids,
        max_sequence_length=MAX_SEQUENCE_LENGTH - prefix_ids.shape[1],
        eos_token_id=tokenizer.eos_token_id,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        generator=generator,
    )
    smiles_list = []
    for ids in generated_ids[:, prefix_ids.shape[1] :].tolist():
        if tokenizer.eos_token_id in ids:  # cut at the first <eos>
            ids = ids[: ids.index(tokenizer.eos_token_id)]
        smiles_list.append(tokenizer.decode(ids))
    return smiles_list


def predict_logp(smiles: list[str]) -> np.ndarray:
    """Return the Hyformer-predicted logP of each SMILES."""
    batch = PredictionCollator(tokenizer, max_length=config.max_seq_len)([{"sequence": s} for s in smiles])
    return model.predict(batch.input_ids.to(device), batch.attention_mask.to(device)).squeeze(-1).cpu().numpy()


def accept_best_of_n(smiles: list[str], target: float, epsilon: float = 0.5) -> list[str]:
    """Best-of-n: the single SMILES whose predicted logP is closest to `target` (`epsilon` is unused)."""
    predictions = predict_logp(smiles)
    best = int(np.argmin(np.abs(predictions - target)))
    return [smiles[best]]


def accept_rejection(smiles: list[str], target: float, epsilon: float = 0.5) -> list[str]:
    """Rejection sampling: every SMILES whose predicted logP is within `epsilon` of `target`."""
    predictions = predict_logp(smiles)
    return [s for s, pred in zip(smiles, predictions) if abs(pred - target) <= epsilon]


# 1. Generate candidates (shared by both methods); validity is only checked at evaluation
candidates = generate_candidates(N_CANDIDATES)

print(f"Target logP: {TARGET_LOGP:.2f} (epsilon={EPSILON})")
print(f"generated {N_CANDIDATES} candidates")

# 2. Select from all candidates with each accept_fn
ACCEPT_FNS = {"best-of-n": accept_best_of_n, "rejection": accept_rejection}
selected = {name: fn(candidates, TARGET_LOGP, EPSILON) for name, fn in ACCEPT_FNS.items()}

for name, smiles_list in selected.items():
    print(f"{name}: accepted {len(smiles_list)} of {len(candidates)} candidates")


In [ ]:
# Evaluate all candidates and the molecules kept by each method: mean absolute deviation from the target, std and validity
def evaluate(smiles_list: list[str], target: float) -> dict[str, float]:
    """Return the validity of `smiles_list` and, over its valid molecules, the mean absolute deviation of the
    RDKit logP from `target` and the standard deviation of the RDKit logP."""
    valid = [s for s in smiles_list if calculate_validity(s)]
    logp = np.array([calculate_logp(s) for s in valid])
    return {
        "validity": len(valid) / len(smiles_list) if smiles_list else 0.0,
        "mad": float(np.abs(logp - target).mean()) if valid else float("nan"),
        "std": float(logp.std()) if valid else float("nan"),
    }

# logP is computed with RDKit, not the model
for name, smiles_list in {"all candidates": candidates, **selected}.items():
    metrics = evaluate(smiles_list, TARGET_LOGP)
    print(f"{name:>14}: MAD={metrics['mad']:.3f}  std={metrics['std']:.3f}  validity={metrics['validity']:.1%}")


In [ ]:
# Scatter plots of real (RDKit) vs predicted (Hyformer) logP of the valid candidates, one per accept_fn, accepted ones in red
valid_candidates = [s for s in candidates if calculate_validity(s)]  # RDKit cannot score invalid SMILES

if not valid_candidates:
    print("No valid candidates to plot")
else:
    real_logp = np.array([calculate_logp(s) for s in valid_candidates])
    predictions = predict_logp(valid_candidates)
    lims = [min(real_logp.min(), predictions.min(), TARGET_LOGP - EPSILON), max(real_logp.max(), predictions.max(), TARGET_LOGP + EPSILON)]

    fig, axes = plt.subplots(1, len(selected), figsize=(5 * len(selected), 5), sharex=True, sharey=True)
    for ax, (name, smiles_list) in zip(axes, selected.items()):
        is_accepted = np.array([s in set(smiles_list) for s in valid_candidates])
        ax.scatter(real_logp[~is_accepted], predictions[~is_accepted], alpha=0.7, label="valid candidates")
        ax.scatter(real_logp[is_accepted], predictions[is_accepted], color="red", label=f"valid accepted candidates (n={is_accepted.sum()})")
        ax.plot(lims, lims, color="gray", linestyle=":", label="y = x")  # perfect prediction
        ax.axvline(TARGET_LOGP, color="black", linestyle="--", label=f"target (true logP) = {TARGET_LOGP:.2f}")
        if name == "best-of-n":
            best_pred = float(predict_logp(smiles_list)[0])  # the selected molecule may be invalid, so it can be missing above
            ax.axhline(best_pred, color="red", linestyle="--", label=f"predicted logP of the best = {best_pred:.2f}")
        else:
            ax.axhspan(TARGET_LOGP - EPSILON, TARGET_LOGP + EPSILON, color="gray", alpha=0.2, label="acceptance region (predicted logP)")
        ax.set_xlabel("True logP (RDKit)")
        ax.set_ylabel("Predicted logP (Hyformer)")
        ax.set_title(name)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


**Task 1.** Change `TRAIN_DATASET` from `"random_1k"` to `"low_logp_1k"` and re-run all the computation. What did you observe?  